# 02 — Portal Traversal

A **portal** governs data flow between two holons. Portals are
first-class RDF entities stored in boundary graphs, discoverable via
SPARQL, with their CONSTRUCT queries attached as literals.

This notebook covers:

1. Registering a portal with a SPARQL CONSTRUCT query
2. Discovering portals via `find_portals_from`/`find_portals_to`
3. Traversing a portal and injecting the result into a target interior
4. Governed traversal with membrane validation and PROV-O provenance
5. Multi-hop path finding


## Setup

Two holons: a CRM system (source) and a company directory (target).

In [ ]:
from holonic import HolonicDataset

ds = HolonicDataset()

ds.add_holon("urn:holon:crm", "Customer CRM")
ds.add_interior("urn:holon:crm", '''
    @prefix crm: <urn:crm:> .
    <urn:contact:001> a crm:Contact ;
        crm:fullName "Alice Chen" ;
        crm:email "alice@example.com" ;
        crm:city "San Francisco" .
    <urn:contact:002> a crm:Contact ;
        crm:fullName "Bob Martinez" ;
        crm:email "bob@example.com" ;
        crm:city "Portland" .
''')

ds.add_holon("urn:holon:directory", "Company Directory")
ds.add_boundary("urn:holon:directory", '''
    @prefix sh:     <http://www.w3.org/ns/shacl#> .
    @prefix schema: <https://schema.org/> .
    @prefix xsd:    <http://www.w3.org/2001/XMLSchema#> .

    <urn:shapes:PersonShape> a sh:NodeShape ;
        sh:targetClass schema:Person ;
        sh:property [ sh:path schema:name ;  sh:minCount 1 ; sh:datatype xsd:string ] ;
        sh:property [ sh:path schema:email ; sh:minCount 1 ; sh:datatype xsd:string ] .
''')

print(f"Holons: {len(ds.list_holons())}")

## Register a transform portal

The portal's CONSTRUCT query is stored in the boundary graph as a
literal. It reshapes source-vocab data into target-vocab data.


In [ ]:
construct = '''
    PREFIX crm:    <urn:crm:>
    PREFIX schema: <https://schema.org/>

    CONSTRUCT {
        ?person a schema:Person ;
            schema:name  ?name ;
            schema:email ?email ;
            schema:workLocation ?city .
    }
    WHERE {
        ?contact a crm:Contact ;
            crm:fullName ?name ;
            crm:email    ?email ;
            crm:city     ?city .
        BIND(IRI(CONCAT("urn:person:", ENCODE_FOR_URI(?email))) AS ?person)
    }
'''

ds.add_portal(
    "urn:portal:crm-to-directory",
    source_iri="urn:holon:crm",
    target_iri="urn:holon:directory",
    construct_query=construct,
    label="CRM → Directory",
)

portals = ds.find_portals_from("urn:holon:crm")
print(f"Portals from CRM: {len(portals)}")
print(f"  {portals[0].label} → {portals[0].target_iri}")

## Discover portals via SPARQL

Portal discovery is always SPARQL-driven — not Python iteration over
cached lists.


In [ ]:
from_crm = ds.find_portals_from("urn:holon:crm")
for p in from_crm:
    print(f"{p.iri}  →  {p.target_iri}")


## Traverse and inject

`traverse_portal()` runs the CONSTRUCT against the dataset and
optionally appends the result into a target graph. Below we inject
into the directory holon's interior.


In [ ]:
target_interior = "urn:holon:directory/interior"
projected = ds.traverse_portal(
    "urn:portal:crm-to-directory",
    inject_into=target_interior,
)
print(f"projected triples: {len(projected)}")


## Governed traversal with validation + provenance

`traverse()` composes discovery, traversal, membrane validation, and
`prov:Activity` recording as one governed operation. If validation
fails, data is NOT injected.


In [ ]:
# Clear the directory interior first so we can rerun cleanly
ds.backend.delete_graph("urn:holon:directory/interior")

projected, membrane = ds.traverse(
    "urn:holon:crm",
    "urn:holon:directory",
    validate=True,
    agent_iri="urn:agent:directory-pipeline",
)
print(f"membrane health: {membrane.health}")
print(f"violations:      {len(membrane.violations)}")


## Inspect the provenance trail

Every governed traversal writes a `prov:Activity` into the target
holon's context graph. The activity links back to the source and
target holons and records the agent and timestamp.


In [ ]:
rows = ds.query('''
    PREFIX prov: <http://www.w3.org/ns/prov#>
    SELECT ?activity ?start ?agent
    WHERE {
        GRAPH ?g {
            ?activity a prov:Activity ;
                prov:used <urn:holon:crm> ;
                prov:generated <urn:holon:directory> ;
                prov:startedAtTime ?start ;
                prov:wasAssociatedWith ?agent .
        }
    }
    ORDER BY DESC(?start)
''')
rows


## Multi-hop path finding

Given an intermediate holon, the library finds a path from any source
to any reachable target via a single SPARQL query.


In [ ]:
ds.add_holon("urn:holon:analytics", "Analytics Warehouse")
ds.add_portal(
    "urn:portal:directory-to-analytics",
    source_iri="urn:holon:directory",
    target_iri="urn:holon:analytics",
    construct_query="CONSTRUCT { ?s ?p ?o } WHERE { ?s ?p ?o }",
    label="Directory → Analytics",
)

path = ds.find_path("urn:holon:crm", "urn:holon:analytics")
print([p.iri for p in path])


## Where to go next

- `04_cco_to_schemaorg` — cross-vocabulary portals at scale
- `08_scope_resolution` — find holons by predicate across the holarchy
- `10_projection_plugins` — projection pipelines as RDF-declared specs
